# ADS1002 Group Project -- Data Preprocessing

This notebook covers ingestion, structural cleaning, and merging of the AEMO demand data and BoM temperature data. Data cleaning (missing values, quality flags) and EDA specific to the group's research question will be handled in a separate notebook.

## 1. Imports

In [1]:
import pandas as pd
import glob
import os

Standard libraries for reading, combining, and cleaning tabular data. No plotting/modelling libraries needed yet -- those get added once EDA begins in the follow-up notebook.

## 2. Demand Data Processing

In [2]:
DEMAND_DIR = "/Users/thish/Documents/ADS1002/Datasets for temperature and energy demand modelling-20260731/ALL_DEMAND_DATA"

# filters out __MACOSX/.DS_Store/._ junk from Mac zip/unzip
csv_demand = glob.glob(os.path.join(DEMAND_DIR, "*.csv"))
csv_demand = [f for f in csv_demand if not os.path.basename(f).startswith("._")]

print(f"Found {len(csv_demand)} demand CSV files")

Found 1136 demand CSV files


Locate all monthly AEMO demand CSVs (2000-2019, one file per state per month). Files starting with `._` are Mac metadata artefacts left over from zipping/unzipping and are filtered out, as they aren't real data files and would break the read step below.

In [3]:
df_demand_list = [pd.read_csv(file) for file in csv_demand]
combined_demand = pd.concat(df_demand_list, ignore_index=True)

Read every monthly file into its own DataFrame and stack them into one long-format table (one row per state per half-hour). This "long" structure -- rather than separate tables per state -- makes later cleaning, merging, and groupby-based EDA much simpler, since the same logic only needs to be written once.

In [4]:
combined_demand.head()

,REGION,SETTLEMENTDATE,TOTALDEMAND,RRP,PERIODTYPE
0,NSW1,2000/09/01 00:30,8117.23667,36.72,TRADE
1,NSW1,2000/09/01 01:00,7799.70000,32.92,TRADE
2,NSW1,2000/09/01 01:30,7453.99000,27.84,TRADE
3,NSW1,2000/09/01 02:00,7095.55333,30.62,TRADE
4,NSW1,2000/09/01 02:30,6786.22167,33.53,TRADE


In [5]:
combined_demand["SETTLEMENTDATE"] = pd.to_datetime(
    combined_demand["SETTLEMENTDATE"], format="mixed", errors="coerce"
)

print("Unparseable dates:", combined_demand["SETTLEMENTDATE"].isna().sum())

Unparseable dates: 0


Convert `SETTLEMENTDATE` from text to a proper datetime. `format="mixed"` is required because the raw files aren't consistent -- earlier files omit seconds ("2000/01/01 00:30") while later files include them ("2009/09/01 00:30:00"); a single fixed format string would silently fail to parse whichever style it wasn't given.

In [6]:
region_to_state = {
    "NSW1": "NSW",
    "VIC1": "VIC",
    "QLD1": "QLD",
    "SA1": "SA",
    "TAS1": "TAS",
}
combined_demand["state"] = combined_demand["REGION"].map(region_to_state)

print("Unmapped regions:", combined_demand["state"].isna().sum())

Unmapped regions: 0


Derive a clean state code (e.g. "NSW") from AEMO's region code (e.g. "NSW1"). This becomes the join key used later to merge with the temperature dataset, which is keyed by state rather than by AEMO region. `state` is dropped again after the merge (see Section 4) since `REGION` alone is sufficient to reconstruct it -- kept here purely as the merge key.

In [7]:
print("Combined shape:", combined_demand.shape)
print("\nRows per region:")
print(combined_demand["REGION"].value_counts())
print("\nDate range:", combined_demand["SETTLEMENTDATE"].min(), "to", combined_demand["SETTLEMENTDATE"].max())
print("\nMissing values per column:")
print(combined_demand.isna().sum())

combined_demand.head()

Combined shape: (1658965, 6)

Rows per region:
REGION
NSW1    350632
VIC1    350632
SA1     350632
QLD1    350632
TAS1    256437
Name: count, dtype: int64

Date range: 2000-01-01 00:30:00 to 2020-01-01 00:00:00

Missing values per column:
REGION            0
SETTLEMENTDATE    0
TOTALDEMAND       0
RRP               0
PERIODTYPE        0
state             0
dtype: int64


,REGION,SETTLEMENTDATE,TOTALDEMAND,RRP,PERIODTYPE,state
0,NSW1,2000-09-01 00:30:00,8117.23667,36.72,TRADE,NSW
1,NSW1,2000-09-01 01:00:00,7799.70000,32.92,TRADE,NSW
2,NSW1,2000-09-01 01:30:00,7453.99000,27.84,TRADE,NSW
3,NSW1,2000-09-01 02:00:00,7095.55333,30.62,TRADE,NSW
4,NSW1,2000-09-01 02:30:00,6786.22167,33.53,TRADE,NSW


Sanity checks confirm: no unmapped regions, plausible row counts per state, and a date range matching the expected 2000-2019 window. TAS is expected to have fewer rows than the other states, since Tasmania only joined the NEM part-way through 2005.

## 3. Temperature Data Processing

In [8]:
TEMP_DIR = "/Users/thish/Documents/ADS1002/Datasets for temperature and energy demand modelling-20260731/Temperature Data"
STN_DETAILS_PATH = "/Users/thish/Documents/ADS1002/Datasets for temperature and energy demand modelling-20260731/HM01X_StnDet_999999999743964.txt"

txt_temperature = glob.glob(os.path.join(TEMP_DIR, "*.txt"))
txt_temperature = [f for f in txt_temperature if not os.path.basename(f).startswith("._")]

print(f"Found {len(txt_temperature)} temperature station files:")
for f in txt_temperature:
    print(" -", os.path.basename(f))

Found 6 temperature station files:
 - HM01X_Data_094029_999999999743964.txt
 - HM01X_Data_086338_999999999743964.txt
 - HM01X_Data_023090_999999999743964.txt
 - HM01X_Data_066062_999999999743964.txt
 - HM01X_Data_086071_999999999743964.txt
 - HM01X_Data_040913_999999999743964.txt


Locate the 6 BoM station observation files, filtering out Mac metadata artefacts as above.

In [9]:
# from BoM Notes file -- 34 fields, 1 header/description line before data starts
TEMP_COLUMNS = [
    "record_id", "station_number",
    "year_local", "month_local", "day_local", "hour_local", "minute_local",
    "year_std", "month_std", "day_std", "hour_std", "minute_std",
    "precipitation_9am_mm", "precipitation_quality",
    "air_temperature_c", "air_temperature_quality",
    "wet_bulb_temp_c", "wet_bulb_quality",
    "dew_point_temp_c", "dew_point_quality",
    "relative_humidity_pct", "relative_humidity_quality",
    "wind_speed_kmh", "wind_speed_quality",
    "wind_direction_deg", "wind_direction_quality",
    "max_gust_kmh", "max_gust_quality",
    "mslp_hpa", "mslp_quality",
    "station_level_pressure_hpa", "station_level_pressure_quality",
    "aws_flag", "end_marker",
]

Column names for the raw BoM files, taken from the accompanying Notes/data-dictionary file -- 34 fixed fields per row, since the files have no usable header row of their own (the literal first line is a human-readable description with embedded commas, not real columns).

In [10]:
df_temp_list = []
for file in txt_temperature:
    df = pd.read_csv(
        file, skiprows=1, header=None, names=TEMP_COLUMNS,
        na_values=["", " "], skipinitialspace=True, dtype=str,
    )
    df_temp_list.append(df)

combined_temp = pd.concat(df_temp_list, ignore_index=True)
combined_temp = combined_temp.drop(columns=["record_id", "end_marker"])

Skip the 1-line description header on each file and combine all 6 stations into one long-format table. Read as strings initially rather than letting pandas infer dtypes, to avoid the mixed-dtype warnings that come from blank/missing values in numeric columns.

In [11]:
numeric_cols = [
    "station_number",
    "year_local", "month_local", "day_local", "hour_local", "minute_local",
    "year_std", "month_std", "day_std", "hour_std", "minute_std",
    "precipitation_9am_mm", "air_temperature_c", "wet_bulb_temp_c",
    "dew_point_temp_c", "relative_humidity_pct", "wind_speed_kmh",
    "wind_direction_deg", "max_gust_kmh", "mslp_hpa",
    "station_level_pressure_hpa", "aws_flag",
]
for col in numeric_cols:
    combined_temp[col] = pd.to_numeric(combined_temp[col], errors="coerce")

# quality flag columns (Y/N/W/S/I) stay as strings, fix "nan"-string bug
quality_cols = [c for c in combined_temp.columns if c.endswith("_quality")]
for col in quality_cols:
    combined_temp[col] = combined_temp[col].astype(str).str.strip().replace("nan", pd.NA)

Convert measurement columns to numeric and quality-flag columns (Y/N/W/S/I) to clean strings. The `.replace("nan", pd.NA)` step corrects a side effect of `.astype(str)`, which otherwise turns genuinely missing quality flags into the literal text "nan" rather than a true missing value.

In [12]:
# using LOCAL STANDARD TIME fields
combined_temp["datetime_std"] = pd.to_datetime(
    combined_temp[["year_std", "month_std", "day_std", "hour_std", "minute_std"]]
    .rename(columns={
        "year_std": "year", "month_std": "month", "day_std": "day",
        "hour_std": "hour", "minute_std": "minute",
    }),
    errors="coerce",
)

Build a datetime column from the `*_std` (local standard time) fields rather than the local time fields, to avoid the daylight-saving discontinuities that would otherwise distort half-hourly time alignment.

In [13]:
STN_COLUMNS = [
    "record_id", "station_number", "rainfall_district_code", "station_name",
    "date_opened", "date_closed", "latitude", "longitude", "location_method",
    "state", "height_station_m", "height_barometer_m", "wmo_index",
    "first_year", "last_year", "pct_complete",
    "pct_flag_Y", "pct_flag_N", "pct_flag_W", "pct_flag_S", "pct_flag_I",
    "end_marker",
]

Column names for the station details/metadata file, taken from the same BoM Notes file's byte-position spec.

In [14]:
stn_details = pd.read_csv(
    STN_DETAILS_PATH, skiprows=5, header=None,
    names=STN_COLUMNS, skipinitialspace=True,
)
stn_details["station_name"] = stn_details["station_name"].str.strip()
stn_details["state"] = stn_details["state"].str.strip()
stn_details = stn_details.drop(columns=["record_id", "end_marker", "location_method"])

Read the station metadata file (5 header/note lines skipped) to get each station's name, state, and coordinates for merging onto the observation data.

In [15]:
stn_details.head(6)

,station_number,rainfall_district_code,station_name,date_opened,date_closed,latitude,longitude,state,height_station_m,height_barometer_m,wmo_index,first_year,last_year,pct_complete,pct_flag_Y,pct_flag_N,pct_flag_W,pct_flag_S,pct_flag_I
0,86338,86,MELBOURNE (OLYMPIC PARK),05/2013,NaN,-37.8255,144.9816,VIC,7.5,7.5,95936,2013,2020,103,0,100,0,0,0
1,86071,86,MELBOURNE REGIONAL OFFICE,01/1908,01/2015,-37.8075,144.9700,VIC,31.2,32.2,94868,2000,2015,99,0,100,0,0,0
2,23090,23A,ADELAIDE (KENT TOWN),01/1977,NaN,-34.9211,138.6216,SA,48.0,51.0,94675,2000,2020,101,0,100,0,0,0
3,66062,66,SYDNEY (OBSERVATORY HILL),01/1858,NaN,-33.8607,151.2050,NSW,39.0,40.2,94768,2000,2020,99,0,100,0,0,0
4,94029,94,HOBART (ELLERSLIE ROAD),01/1882,NaN,-42.8897,147.3278,TAS,50.5,51.4,94970,2000,2020,110,0,100,0,0,0
5,40913,40,BRISBANE,12/1999,NaN,-27.4808,153.0389,QLD,8.1,8.3,94576,2000,2020,99,0,100,0,0,0


In [16]:
combined_temp = combined_temp.merge(
    stn_details[["station_number", "station_name", "state", "latitude", "longitude"]],
    on="station_number", how="left",
)

print("Unmatched rows:", combined_temp["station_name"].isna().sum())

Unmatched rows: 0


Left merge attaches station name, state, and coordinates onto every temperature observation, keyed by `station_number`. 0 unmatched rows confirms all 6 stations matched successfully.

In [17]:
# Regional Office (086071) closes 01/2015, Olympic Park (086338) opens 05/2013
# -- cutover at Olympic Park's opening, since it's BoM's official replacement site
MELBOURNE_CUTOVER = pd.Timestamp("2013-05-01")

is_vic = combined_temp["state"] == "VIC"
keep_86071 = is_vic & (combined_temp["station_number"] == 86071) & (combined_temp["datetime_std"] < MELBOURNE_CUTOVER)
keep_86338 = is_vic & (combined_temp["station_number"] == 86338) & (combined_temp["datetime_std"] >= MELBOURNE_CUTOVER)
keep_non_vic = ~is_vic

combined_temp = combined_temp[keep_86071 | keep_86338 | keep_non_vic].copy()

print("Duplicate state/timestamp rows remaining:", combined_temp.duplicated(subset=["state", "datetime_std"]).sum())

Duplicate state/timestamp rows remaining: 0


Melbourne has two overlapping stations -- Regional Office (086071, closes 01/2015) and Olympic Park (086338, opens 05/2013) -- which would otherwise produce duplicate VIC readings for the same timestamp during their ~20-month overlap. Resolved with a cutover at Olympic Park's opening date, since it is BoM's official replacement station. 0 duplicate state/timestamp rows confirms the fix worked.

In [18]:
print("Combined shape:", combined_temp.shape)
print("\nRows per state:")
print(combined_temp["state"].value_counts())
print("\nAny unparseable datetimes?", combined_temp["datetime_std"].isna().sum())
print("\nMissing values per column:")
print(combined_temp.isna().sum())

combined_temp.head()

Combined shape: (1796707, 37)

Rows per state:
state
TAS    386362
SA     355906
VIC    352517
QLD    350977
NSW    350945
Name: count, dtype: int64

Any unparseable datetimes? 0

Missing values per column:
station_number                         0
year_local                             0
month_local                            0
day_local                              0
hour_local                             0
minute_local                           0
year_std                               0
month_std                              0
day_std                                0
hour_std                               0
minute_std                             0
precipitation_9am_mm               63715
precipitation_quality              63715
air_temperature_c                   1215
air_temperature_quality             1215
wet_bulb_temp_c                     4275
wet_bulb_quality                    4275
dew_point_temp_c                    1503
dew_point_quality                   1503
relative_humid

,station_number,year_local,month_local,day_local,hour_local,minute_local,year_std,month_std,day_std,hour_std,...,mslp_hpa,mslp_quality,station_level_pressure_hpa,station_level_pressure_quality,aws_flag,datetime_std,station_name,state,latitude,longitude
0,94029,2000,1,1,2,0,2000,1,1,1,...,1019.3,N,1013.0,N,NaN,2000-01-01 01:00:00,HOBART (ELLERSLIE ROAD),TAS,-42.8897,147.3278
1,94029,2000,1,1,2,30,2000,1,1,1,...,1019.1,N,1012.8,N,NaN,2000-01-01 01:30:00,HOBART (ELLERSLIE ROAD),TAS,-42.8897,147.3278
2,94029,2000,1,1,3,0,2000,1,1,2,...,1018.9,N,1012.6,N,NaN,2000-01-01 02:00:00,HOBART (ELLERSLIE ROAD),TAS,-42.8897,147.3278
3,94029,2000,1,1,3,30,2000,1,1,2,...,1018.7,N,1012.4,N,NaN,2000-01-01 02:30:00,HOBART (ELLERSLIE ROAD),TAS,-42.8897,147.3278
4,94029,2000,1,1,4,0,2000,1,1,3,...,1018.5,N,1012.2,N,NaN,2000-01-01 03:00:00,HOBART (ELLERSLIE ROAD),TAS,-42.8897,147.3278


Final structural sanity check on the temperature side before merging with demand: expected row counts per state, no unparseable datetimes, and a visible picture of remaining missing values (to be addressed in the cleaning notebook, not here).

## 4. Merging Demand and Temperature

In [19]:
merged_df = pd.merge(
    combined_demand,
    combined_temp,
    left_on=["state", "SETTLEMENTDATE"],
    right_on=["state", "datetime_std"],
    how="left",   # keep every demand row, even if no matching temperature reading exists
)

print("Merged shape:", merged_df.shape)

Merged shape: (1658965, 42)


Left merge combines demand and temperature on `(state, timestamp)` together -- not timestamp alone -- so each state's demand is matched only against its own temperature, rather than against whichever station happens to share that half-hour. `left` keeps every demand row, since demand is the primary dataset.

In [20]:
#double checking for errors in dataset when merging
print("Demand rows before merge:", len(combined_demand))
print("Merged rows after merge:", len(merged_df))

print("\nRows with no matching temperature reading:")
print(merged_df["datetime_std"].isna().sum())

print("\nMissing values per column:")
print(merged_df.isna().sum())

merged_df.head()

Demand rows before merge: 1658965
Merged rows after merge: 1658965

Rows with no matching temperature reading:
7682

Missing values per column:
REGION                                 0
SETTLEMENTDATE                         0
TOTALDEMAND                            0
RRP                                    0
PERIODTYPE                             0
state                                  0
station_number                      7682
year_local                          7682
month_local                         7682
day_local                           7682
hour_local                          7682
minute_local                        7682
year_std                            7682
month_std                           7682
day_std                             7682
hour_std                            7682
minute_std                          7682
precipitation_9am_mm               18854
precipitation_quality              18854
air_temperature_c                   8809
air_temperature_quality             

,REGION,SETTLEMENTDATE,TOTALDEMAND,RRP,PERIODTYPE,state,station_number,year_local,month_local,day_local,...,max_gust_quality,mslp_hpa,mslp_quality,station_level_pressure_hpa,station_level_pressure_quality,aws_flag,datetime_std,station_name,latitude,longitude
0,NSW1,2000-09-01 00:30:00,8117.23667,36.72,TRADE,NSW,66062.0,2000.0,9.0,1.0,...,<NA>,1011.3,N,1006.5,N,1.0,2000-09-01 00:30:00,SYDNEY (OBSERVATORY HILL),-33.8607,151.205
1,NSW1,2000-09-01 01:00:00,7799.70000,32.92,TRADE,NSW,66062.0,2000.0,9.0,1.0,...,<NA>,1011.1,N,1006.3,N,1.0,2000-09-01 01:00:00,SYDNEY (OBSERVATORY HILL),-33.8607,151.205
2,NSW1,2000-09-01 01:30:00,7453.99000,27.84,TRADE,NSW,66062.0,2000.0,9.0,1.0,...,<NA>,1011.3,N,1006.5,N,1.0,2000-09-01 01:30:00,SYDNEY (OBSERVATORY HILL),-33.8607,151.205
3,NSW1,2000-09-01 02:00:00,7095.55333,30.62,TRADE,NSW,66062.0,2000.0,9.0,1.0,...,<NA>,1011.3,N,1006.5,N,1.0,2000-09-01 02:00:00,SYDNEY (OBSERVATORY HILL),-33.8607,151.205
4,NSW1,2000-09-01 02:30:00,6786.22167,33.53,TRADE,NSW,66062.0,2000.0,9.0,1.0,...,<NA>,1011.2,N,1006.4,N,1.0,2000-09-01 02:30:00,SYDNEY (OBSERVATORY HILL),-33.8607,151.205


Row count is unchanged from `combined_demand` (confirms the merge added columns, not rows). The unmatched-temperature count is investigated properly in the next cell rather than assumed.

In [21]:
mismatches = merged_df[merged_df["SETTLEMENTDATE"] != merged_df["datetime_std"]]

print("Total mismatched rows:", len(mismatches))
print("\nOf these, how many have a completely missing datetime_std (no match at all)?")
print(mismatches["datetime_std"].isna().sum())

print("\nMismatches by state:")
print(mismatches["state"].value_counts())

print("\nDate range of mismatched rows:")
print(mismatches["SETTLEMENTDATE"].min(), "to", mismatches["SETTLEMENTDATE"].max())

Total mismatched rows: 7682

Of these, how many have a completely missing datetime_std (no match at all)?
7682

Mismatches by state:
state
VIC    2486
TAS    1471
SA     1455
QLD    1338
NSW     932
Name: count, dtype: int64

Date range of mismatched rows:
2000-01-01 08:00:00 to 2019-10-08 08:00:00


7,682 rows (~0.5%) have no matching temperature reading, spread across all five states roughly in proportion -- not concentrated in Melbourne, ruling out the station cutover as the cause. This is a normal, small coverage gap for 20 years of automatic weather station data. It is left as-is here and will be addressed explicitly (drop vs. impute) in the cleaning notebook, not silently dropped in preprocessing.

In [22]:
# datetime_std is a duplicate of SETTLEMENTDATE (identical values from the merge key).
# state is dropped since REGION (e.g. "NSW1") already encodes the same information
# and can be re-derived with the region_to_state mapping above whenever needed.
merged_df = merged_df.drop(columns=["datetime_std", "state"])

Two redundant columns are removed: `datetime_std` duplicates `SETTLEMENTDATE`, and `state` duplicates the information already in `REGION`. Keeping one canonical source for each (`SETTLEMENTDATE` for time, `REGION` for location) avoids carrying two columns that must always agree -- `state` can be regenerated from `REGION` with a one-line mapping whenever a future step needs it (e.g. grouping by state in EDA, or joining the rooftop PV dataset).

In [23]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1658965 entries, 0 to 1658964
Data columns (total 40 columns):
 #   Column                          Non-Null Count    Dtype         
---  ------                          --------------    -----         
 0   REGION                          1658965 non-null  object        
 1   SETTLEMENTDATE                  1658965 non-null  datetime64[ns]
 2   TOTALDEMAND                     1658965 non-null  float64       
 3   RRP                             1658965 non-null  float64       
 4   PERIODTYPE                      1658965 non-null  object        
 5   station_number                  1651283 non-null  float64       
 6   year_local                      1651283 non-null  float64       
 7   month_local                     1651283 non-null  float64       
 8   day_local                       1651283 non-null  float64       
 9   hour_local                      1651283 non-null  float64       
 10  minute_local                    1651283 no

Final structure check on the merged table before export -- confirms column count, dtypes, and non-null counts all look as expected.

## 5. Export

In [25]:
#merged_df.to_csv("merged_df_raw.csv.gz", index=False, compression="gzip")

Saved as `merged_df_raw.csv.gz` -- "raw" because this reflects the merged data before any missing-value handling. A cleaned version (with the ~7,682 unmatched rows resolved) will be produced in the follow-up cleaning/EDA notebook, to avoid ambiguity over which file a teammate is working from.

## Next Steps

This notebook covers structural ingestion and merging only. The next notebook (cleaning + EDA, specific to the group's chosen research question) will handle:
- Deciding how to treat the ~7,682 rows with no matching temperature reading
- Deciding how to treat BoM quality flags (`W`/`S`/`I`) on temperature variables
- Any further column selection/feature engineering specific to the research question being addressed